# 01 Setup and Processing

prepares the **FakeHealth + healthfact** dataset for model training, It reads the raw combined CSV, performs light cleaning, keeps only binary-labeled rows, and writes the processed files back to the project folder.


In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd


In [2]:
PROJECT_ROOT = Path(r'C:\Users\ribam\Desktop\Reseach\Dataset')
DATA_ROOT = PROJECT_ROOT / 'dataset'
RAW_DIR = DATA_ROOT / 'raw'
PROCESSED_DIR = DATA_ROOT / 'processed'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

RAW_PATH = RAW_DIR / 'fakehealth_healthfact_combined.csv'
PROCESSED_PATH = PROCESSED_DIR / 'fakehealth_healthfact_binary_clean.csv'
SUMMARY_PATH = PROCESSED_DIR / 'fakehealth_healthfact_processing_summary.json'

RAW_PATH, PROCESSED_PATH, SUMMARY_PATH


(WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/dataset/raw/fakehealth_healthfact_combined.csv'),
 WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/dataset/processed/fakehealth_healthfact_binary_clean.csv'),
 WindowsPath('C:/Users/ribam/Desktop/Reseach/Dataset/dataset/processed/fakehealth_healthfact_processing_summary.json'))

In [3]:
df = pd.read_csv(RAW_PATH)
print('Raw shape:', df.shape)
df.head(3)


Raw shape: (14422, 11)


,dataset,split,record_id,text_kind,title,text,url,source,publish_date,label_original,label_binary
0,fakehealth,release,news_reviews_00000,article_text,Tiny implantable device short-circuits hunger ...,"MADISON, Wis. -- More than 700 million adults ...",https://web.archive.org/web/20181218015531/htt...,https://web.archive.org,1.546060e+09,2,0.0
1,fakehealth,release,news_reviews_00001,article_text,Scientists report CRISPR restores effectivenes...,"Wilmington, DE, December 17, 2018 - The CRISPR...",https://web.archive.org/web/20181217203805/htt...,https://web.archive.org,1.546060e+09,3,1.0
2,fakehealth,release,news_reviews_00002,article_text,Probiotics could help millions of patients suf...,About 3 million people in the US are diagnosed...,https://web.archive.org/web/20181213085845/htt...,https://web.archive.org,1.546060e+09,1,0.0


In [4]:
audit = {
    'dataset_counts': df['dataset'].value_counts(dropna=False).to_dict(),
    'split_counts': df['split'].value_counts(dropna=False).to_dict(),
    'text_kind_counts': df['text_kind'].value_counts(dropna=False).to_dict(),
    'label_original_counts': df['label_original'].fillna('').astype(str).value_counts(dropna=False).to_dict(),
    'label_binary_counts': df['label_binary'].fillna('').astype(str).value_counts(dropna=False).to_dict(),
    'missing_text_rows': int(df['text'].isna().sum()),
}
audit


{'dataset_counts': {'healthfact': 12266, 'fakehealth': 2156},
 'split_counts': {'train': 9814,
  'story': 1564,
  'test': 1235,
  'dev': 1217,
  'release': 592},
 'text_kind_counts': {'claim': 12266, 'article_text': 2156},
 'label_original_counts': {'TRUE': 6306,
  'FALSE': 3769,
  'mixture': 1799,
  '3': 707,
  '2': 509,
  '4': 498,
  'unproven': 377,
  '5': 228,
  '1': 183,
  '0': 31,
  '': 15},
 'label_binary_counts': {'1.0': 7542, '0.0': 4689, '': 2191},
 'missing_text_rows': 0}

## Cleaning Rules

The processing below keeps the setup reproducible while making the later models better suited to short claim-style inputs:

- strip repeated whitespace from `title` and `text`
- normalize `label_binary` to only `0` or `1`
- drop rows without a binary label
- keep one row per `dataset + record_id`
- add a numeric `label` column for training
- add `model_text`, which safely combines useful title/context with the claim text
- append a small, clearly marked curated claim set for common demo failure modes
- add basic text-length features for quick EDA

The curated rows are not a replacement for a larger sourced dataset; they are a transparent augmentation/challenge set so the model sees plain-English vaccine, prevention, and public-health wording during training.


In [5]:
def normalize_space(value):
    if pd.isna(value):
        return ''
    value = str(value)
    return re.sub(r'\s+', ' ', value).strip()

def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    text_value = str(value).strip()
    if text_value in {'0', '1'}:
        return text_value

    try:
        numeric_value = float(text_value)
    except ValueError:
        return np.nan

    if numeric_value == 0.0:
        return '0'
    if numeric_value == 1.0:
        return '1'
    return np.nan


In [6]:
CURATED_CLAIMS = [
    # Reliable public-health claims. These improve coverage for plain-English true statements.
    ("train", "COVID-19 vaccines reduce the risk of severe illness, hospitalization, and death.", 1),
    ("train", "Vaccines help the immune system recognize and fight specific infections.", 1),
    ("train", "Washing hands with soap can reduce the spread of infectious diseases.", 1),
    ("train", "Regular physical activity reduces the risk of heart disease.", 1),
    ("train", "A balanced diet rich in fruits and vegetables supports a healthy immune system.", 1),
    ("train", "Quitting smoking lowers the risk of heart disease and lung cancer.", 1),
    ("train", "High blood pressure can increase the risk of stroke and heart disease.", 1),
    ("train", "Wearing sunscreen helps reduce the risk of skin cancer.", 1),
    ("train", "Antibiotics treat bacterial infections, not viral infections such as the common cold.", 1),
    ("train", "Drinking enough water supports hydration, but it does not cure cancer.", 1),
    ("train", "Suicide kills one person every 40 seconds worldwide, according to WHO data.", 1),
    ("train", "FDA approves irritable bowel drug.", 1),
    ("train", "Vitamin D3 might ease menstrual cramps.", 1),
    ("train", "Aspirin may cut cancer deaths, but caution is still needed.", 1),
    ("train", "Ebola vaccines showed promise in clinical studies.", 1),
    ("dev", "Mask wearing can reduce the spread of respiratory infections.", 1),
    ("dev", "Vaccination can reduce severe outcomes from COVID-19.", 1),
    ("dev", "Exercise and a healthy diet can support heart health.", 1),
    ("test", "Vaccines are designed to prevent disease, not to cure an active infection.", 1),
    ("test", "Handwashing can lower the risk of spreading germs.", 1),
    ("test", "Stopping smoking reduces health risks over time.", 1),

    # Misinformation claims. These target common false or misleading health patterns.
    ("train", "Drinking bleach can cure COVID-19 and kills viruses instantly.", 0),
    ("train", "Taking COVID vaccines can cure COVID.", 0),
    ("train", "COVID-19 vaccines are useless.", 0),
    ("train", "Vaccines cause autism in young children.", 0),
    ("train", "5G towers are spreading COVID-19 through electromagnetic radiation.", 0),
    ("train", "Drinking lemon water every morning cures cancer.", 0),
    ("train", "Eating a teaspoon of turmeric daily cures all cancers.", 0),
    ("train", "Antibiotics cure viral infections like COVID-19.", 0),
    ("train", "Detox teas remove toxins and cure chronic diseases.", 0),
    ("train", "A single vitamin supplement prevents all common colds.", 0),
    ("train", "The government is hiding evidence that vaccines cause autism.", 0),
    ("train", "COVID-19 vaccines contain microchips for tracking people.", 0),
    ("train", "Natural remedies always work better than prescribed medicine.", 0),
    ("train", "Cancer can be cured by avoiding sugar completely.", 0),
    ("train", "Masks cause dangerous carbon dioxide poisoning in healthy adults.", 0),
    ("dev", "The COVID vaccine is useless because vaccinated people can still get infected.", 0),
    ("dev", "5G networks spread coronavirus symptoms.", 0),
    ("dev", "Lemon water cures cancer better than chemotherapy.", 0),
    ("test", "Vaccines cause autism and doctors are hiding it.", 0),
    ("test", "Bleach is a safe treatment for COVID-19.", 0),
    ("test", "High doses of vitamin C prevent the common cold entirely.", 0),
]


def build_model_text(row):
    title = normalize_space(row.get("title", ""))
    text = normalize_space(row.get("text", ""))
    if title and title.lower() not in text.lower():
        return f"{title}. {text}"
    return text


df_processed = df.copy()
df_processed["title"] = df_processed["title"].apply(normalize_space)
df_processed["text"] = df_processed["text"].apply(normalize_space)
df_processed["label_binary"] = df_processed["label_binary"].apply(normalize_binary_label)

df_processed = df_processed[df_processed["text"] != ""].copy()
df_processed = df_processed[df_processed["label_binary"].isin(["0", "1"])].copy()
df_processed = df_processed.drop_duplicates(subset=["dataset", "record_id"]).copy()

df_processed["label"] = df_processed["label_binary"].astype(int)
df_processed["model_text"] = df_processed.apply(build_model_text, axis=1)

curated_rows = []
for idx, (split, claim, label) in enumerate(CURATED_CLAIMS, start=1):
    curated_rows.append(
        {
            "dataset": "curated_health_claims",
            "split": split,
            "record_id": f"curated_{idx:04d}",
            "text_kind": "claim",
            "title": "",
            "text": normalize_space(claim),
            "model_text": normalize_space(claim),
            "url": "",
            "source": "curated_research_demo",
            "publish_date": "",
            "label_original": "curated_reliable" if label == 1 else "curated_misinformation",
            "label_binary": str(label),
            "label": int(label),
        }
    )

curated_df = pd.DataFrame(curated_rows)
df_processed = pd.concat([df_processed, curated_df], ignore_index=True)
df_processed = df_processed.drop_duplicates(subset=["dataset", "record_id"]).copy()

df_processed["text_length_chars"] = df_processed["text"].str.len()
df_processed["text_length_words"] = df_processed["text"].str.split().str.len()
df_processed["model_text_length_words"] = df_processed["model_text"].str.split().str.len()

column_order = [
    "dataset",
    "split",
    "record_id",
    "text_kind",
    "title",
    "text",
    "model_text",
    "url",
    "source",
    "publish_date",
    "label_original",
    "label_binary",
    "label",
    "text_length_chars",
    "text_length_words",
    "model_text_length_words",
]
df_processed = df_processed[column_order]

print("Processed shape:", df_processed.shape)
print("Curated rows added:", len(curated_df))
df_processed.head(3)


Processed shape: (12273, 16)
Curated rows added: 42


,dataset,split,record_id,text_kind,title,text,model_text,url,source,publish_date,label_original,label_binary,label,text_length_chars,text_length_words,model_text_length_words
0,fakehealth,release,news_reviews_00000,article_text,Tiny implantable device short-circuits hunger ...,"MADISON, Wis. -- More than 700 million adults ...",Tiny implantable device short-circuits hunger ...,https://web.archive.org/web/20181218015531/htt...,https://web.archive.org,1546059600.0,2,0,0,3615,553,562
1,fakehealth,release,news_reviews_00001,article_text,Scientists report CRISPR restores effectivenes...,"Wilmington, DE, December 17, 2018 - The CRISPR...",Scientists report CRISPR restores effectivenes...,https://web.archive.org/web/20181217203805/htt...,https://web.archive.org,1546059600.0,3,1,1,7022,1136,1145
2,fakehealth,release,news_reviews_00002,article_text,Probiotics could help millions of patients suf...,About 3 million people in the US are diagnosed...,Probiotics could help millions of patients suf...,https://web.archive.org/web/20181213085845/htt...,https://web.archive.org,1546059600.0,1,0,0,2967,453,463


In [7]:
df_processed.groupby(['dataset', 'label_binary']).size().unstack(fill_value=0)


label_binary,0,1
dataset,,
curated_health_claims,21,21
fakehealth,920,1236
healthfact,3769,6306


In [8]:
summary = {
    "raw_rows": int(len(df)),
    "processed_rows": int(len(df_processed)),
    "curated_rows_added": int((df_processed["dataset"] == "curated_health_claims").sum()),
    "dropped_missing_binary": int(df["label_binary"].fillna("").astype(str).eq("").sum()),
    "dataset_counts_processed": df_processed["dataset"].value_counts().to_dict(),
    "label_counts_processed": df_processed["label_binary"].value_counts().to_dict(),
    "split_counts_processed": df_processed["split"].value_counts(dropna=False).to_dict(),
    "model_text_column": "title plus text when the title adds extra context; otherwise text only",
    "curated_note": (
        "curated_health_claims is a small transparent augmentation/challenge set for "
        "plain-English public-health claims. Keep it disclosed separately in reporting."
    ),
}

df_processed.to_csv(PROCESSED_PATH, index=False)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Saved processed CSV to:", PROCESSED_PATH)
print("Saved summary JSON to:", SUMMARY_PATH)
summary


Saved processed CSV to: C:\Users\ribam\Desktop\Reseach\Dataset\dataset\processed\fakehealth_healthfact_binary_clean.csv
Saved summary JSON to: C:\Users\ribam\Desktop\Reseach\Dataset\dataset\processed\fakehealth_healthfact_processing_summary.json


{'raw_rows': 14422,
 'processed_rows': 12273,
 'curated_rows_added': 42,
 'dropped_missing_binary': 2191,
 'dataset_counts_processed': {'healthfact': 10075,
  'fakehealth': 2156,
  'curated_health_claims': 42},
 'label_counts_processed': {'1': 7563, '0': 4710},
 'split_counts_processed': {'train': 8109,
  'story': 1564,
  'dev': 1015,
  'test': 993,
  'release': 592},
 'model_text_column': 'title plus text when the title adds extra context; otherwise text only',
 'curated_note': 'curated_health_claims is a small transparent augmentation/challenge set for plain-English public-health claims. Keep it disclosed separately in reporting.'}

After this, the next notebook is **EDA(Exploratory Data Analysis) and label analysis**, followed by **model preparation / train-test strategy**.
